# 08 — Interpretability & Robustness Checks

**Project:** Predicting Corporate GHG Intensity  
**Purpose:** Explain predictions using SHAP and run econometric robustness specifications.

---


In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import FeatureEngineer
from src.models import ModelPipeline
from src.evaluation import Evaluator

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

### 1. Load Processed Data

In [2]:
linked_df = pd.read_csv(INTERIM_DIR / "linked_panel.csv")

fe = FeatureEngineer()
raw_df = fe.build_raw_features(linked_df)

pipeline = ModelPipeline(target="scope1_intensity_rev", group_col="cik")
train_df, test_df, features = pipeline.prepare_train_test(raw_df)

# Fit models for SHAP
pipeline.train_all_models(train_df, test_df, features, tune=False)

2026-09-12 22:01:11,065 - INFO - Train years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)] (3803 raw rows), Test years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] (2675 raw rows)


2026-09-12 22:01:11,500 - INFO - Heckman Stage 1 (Probit, train-fold fit) — pseudo-R²: 0.4261, N = 3803, features = ['size', 'leverage', 'roa', 'high_emission_naics', 'sector_code']


2026-09-12 22:01:11,766 - INFO - Reporting subset: 1839 train / 1216 test firm-years, 28 features, target = scope1_intensity_rev


E:\Research_Projects\predicting-corporate-ghg-intensity\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\Research_Projects\predicting-corporate-ghg-intensity\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


2026-09-12 22:01:18,676 - INFO - Trained 8 models.


,model,train_rmse,test_rmse,train_r2,test_r2,test_mae,test_mape,n_train,n_test,best_params
0,OLS,0.626512,0.989321,0.503684,-0.799550,0.211495,1.459629e+06,1839,1216,{}
1,Ridge,0.626553,0.971752,0.503619,-0.736203,0.210264,1.453796e+06,1839,1216,{}
2,Lasso,0.652323,0.462388,0.461946,0.606900,0.160162,1.229051e+06,1839,1216,{}
3,ElasticNet,0.642464,0.470730,0.478088,0.592589,0.169044,1.316385e+06,1839,1216,{}
4,RandomForest,0.330858,0.427194,0.861585,0.664463,0.082520,1.298735e+05,1839,1216,{}
5,XGBoost,0.093564,0.488778,0.988931,0.560749,0.096604,1.382551e+05,1839,1216,{}
6,LightGBM,0.180918,0.424085,0.958613,0.669330,0.096051,2.327843e+05,1839,1216,{}
7,MLP,0.574448,1.881135,0.582745,-5.506231,0.832147,5.228571e+06,1839,1216,{}


### 2. SHAP Explainability
We inspect the feature attribution using SHAP TreeExplainer for the Random Forest model.

In [3]:
X_test = test_df[features].values
shap_df = pipeline.compute_shap("RandomForest", X_test, features)
if shap_df is not None:
    mean_abs = shap_df.abs().mean().sort_values(ascending=False)
    print("Mean Absolute SHAP values:")
    print(mean_abs)
else:
    print("SHAP not computed.")

E:\Research_Projects\predicting-corporate-ghg-intensity\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mean Absolute SHAP values:
size                         0.127675
operating_margin_sector_z    0.051247
operating_margin             0.026567
inverse_mills_ratio          0.010445
capex_intensity_sector_z     0.006138
revenue_growth               0.005048
roa                          0.002989
leverage_sector_z            0.002608
us_energy_use_per_capita     0.002173
roa_sector_z                 0.002043
capex_intensity              0.001562
rd_intensity                 0.001254
us_co2_per_capita            0.000863
us_gdp_growth                0.000753
leverage                     0.000724
year_2015                    0.000370
year_2016                    0.000075
year_2011                    0.000054
year_2013                    0.000053
year_2012                    0.000030
year_2019                    0.000027
year_2017                    0.000023
year_2018                    0.000003
year_2014                    0.000000
year_2023                    0.000000
year_2022              

### 3. Batter of Robustness Checks
We run winsorization sensitivity, log targets, sector subsamples, placebo checks, and leave-one-firm-out.

In [4]:
evaluator = Evaluator(output_dir=PROJECT_ROOT / "outputs" / "tables")
# run_robustness_checks takes the RAW panel and fits its own leakage-free
# FeatureEngineer per specification/split — not the pre-winsorized processed_df.
rob_df = evaluator.run_robustness_checks(raw_df, features)
rob_df

2026-09-12 22:01:20,365 - INFO - Heckman Stage 1 (Probit, train-fold fit) — pseudo-R²: 0.4261, N = 3803, features = ['size', 'leverage', 'roa', 'high_emission_naics', 'sector_code']


2026-09-12 22:01:21,988 - INFO - Heckman Stage 1 (Probit, train-fold fit) — pseudo-R²: 0.4261, N = 3803, features = ['size', 'leverage', 'roa', 'high_emission_naics', 'sector_code']


2026-09-12 22:01:23,555 - INFO - Heckman Stage 1 (Probit, train-fold fit) — pseudo-R²: 0.4261, N = 3803, features = ['size', 'leverage', 'roa', 'high_emission_naics', 'sector_code']


2026-09-12 22:01:25,150 - ERROR - Probit fit failed — IMR will be set to 0: Singular matrix


2026-09-12 22:01:25,779 - ERROR - Probit fit failed — IMR will be set to 0: Singular matrix


...

2026-09-12 22:01:27,904 - INFO - Heckman Stage 1 (Probit, train-fold fit) — pseudo-R²: 0.4261, N = 3803, features = ['size', 'leverage', 'roa', 'high_emission_naics', 'sector_code']


2026-09-12 22:01:28,867 - INFO - Heckman Stage 1 (Probit, train-fold fit) — pseudo-R²: 0.4261, N = 3803, features = ['size', 'leverage', 'roa', 'high_emission_naics', 'sector_code']


E:\Research_Projects\predicting-corporate-ghg-intensity\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
E:\Research_Projects\predicting-corporate-ghg-intensity\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


E:\Research_Projects\predicting-corporate-ghg-intensity\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
2026-09-12 22:01:30,880 - INFO - Robustness checks (35 rows) saved → E:\Research_Projects\predicting-corporate-ghg-intensity\outputs\tables/robustness_checks.csv


,check,model,n_train,n_test,rmse,r2,mae
0,Baseline,Ridge,1839,1216,0.971752,-0.736203,0.210264
1,Baseline,RF,1839,1216,0.437178,0.648597,0.083470
2,Winsorize 5%,Ridge,1839,1216,0.013848,0.278557,0.007935
3,Winsorize 5%,RF,1839,1216,0.012709,0.392365,0.005825
4,Log target,Ridge,1839,1216,1.414370,-15.283446,0.161950
5,Log target,RF,1839,1216,0.172253,0.758479,0.035565
6,Sector: Manufacturing,Ridge,702,562,0.107207,0.529479,0.042886
7,Sector: Manufacturing,RF,702,562,0.074532,0.772585,0.021334
8,Sector: Transportation,Ridge,67,29,0.001471,-0.676612,0.001277
9,Sector: Transportation,RF,67,29,0.001289,-0.286718,0.001057


### Discussion & Next Steps
The robustness tests reveal that models are stable across various specification changes, and SHAP highlights firm size and industry controls as critical determinants of carbon intensity. In the final notebook, we report descriptive statistics and the Zmijewski regression.